# SCF convergence threshold and force accuracy

The SCF loop in `pw.x` stops when the change in total energy between two successive
iterations falls below `conv_thr` (Ry).  Because the Hellmann–Feynman forces require
exact Kohn–Sham wavefunctions, any residual error in the electron density propagates
into the computed forces.

`pw.x` estimates this error and prints it alongside the forces:

```
Total force = X.XXXXXX     Total SCF correction = Y.YYYYYY
```

The *total* SCF correction is a vector sum over all atoms and can vanish by symmetry.
With `verbosity = 'medium'`, `pw.x` also prints the correction per atom, allowing a
mean-absolute-error metric that does not cancel.

### Exercises

1. How does the mean absolute SCF force correction scale with `conv_thr`?  Is the
   relationship what you expect from perturbation theory?
2. Compare the SCF correction (QE's own estimate) with the actual force change
   `|ΔFz|` relative to the tightest threshold.  Is the estimate reliable?
3. At what `conv_thr` is the force error below your target accuracy (10 meV/Å)?
   What is the computational cost compared to the default `conv_thr = 1e-6`?

In [ ]:
# ── Bootstrap condacolab (triggers kernel restart on first run) ────────────────
# After the restart, re-run from this cell — it will skip the install.
try:
    import condacolab
    condacolab.check()
    print('✅ condacolab active — continue to next cell')
except Exception:
    import subprocess, sys
    print('Installing condacolab …')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', 'condacolab'],
        stdout=subprocess.DEVNULL,
    )
    import condacolab
    condacolab.install()    # ← kernel restarts here; re-run this cell after restart

In [ ]:
# ── Install QE environment and clone tutorial repo ────────────────────────────
# Takes ~5–10 minutes on first run; skipped automatically on re-run.
import condacolab, subprocess, os, sys, glob
from pathlib import Path
condacolab.check()

ENV_NAME = 'qe_env'
REPO_URL = 'https://github.com/pietrodelugas/qe_with_notebooks.git'
REPO_DIR = '/content/qe_with_notebooks'

# Create qe_env if not already present
_env_bin = f'/usr/local/envs/{ENV_NAME}/bin'
if not os.path.isdir(_env_bin):
    print(f"Creating '{ENV_NAME}' with QE 7.5 — takes ~5–10 minutes …")
    subprocess.run(
        ['conda', 'create', '-n', ENV_NAME, '-c', 'conda-forge', '--yes',
         'python=3.12', 'qe=7.5', 'numpy', 'matplotlib', 'ase', 'scipy'],
        check=True,
    )
    print(f"✅ '{ENV_NAME}' ready")

# Clone tutorial repo (modules + pseudos)
if not os.path.isdir(REPO_DIR):
    print('Cloning tutorial repo …')
    subprocess.run(
        ['git', 'clone', '--depth=1', '--branch', 'distro', REPO_URL, REPO_DIR],
        check=True,
    )
    print('✅ Repo cloned')

# Expose qe_env Python packages to this interpreter
for sp in glob.glob(f'/usr/local/envs/{ENV_NAME}/lib/python*/site-packages'):
    if sp not in sys.path:
        sys.path.insert(0, sp)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verify
print('\nQE executables:')
for exe in ['pw.x', 'bands.x', 'dos.x', 'projwfc.x']:
    p = Path(_env_bin) / exe
    print(f"  {'✅' if p.is_file() else '❌'}  {exe}")
print('\nPython packages:')
for pkg in ['numpy', 'matplotlib', 'ase', 'scipy', 'pw_input']:
    try:
        __import__(pkg)
        print(f'  ✅  {pkg}')
    except ImportError as e:
        print(f'  ❌  {pkg} — {e}')
print('\n🎉 Ready!')

In [ ]:
import os
from pathlib import Path

os.environ['OMP_NUM_THREADS'] = '1'

QE_BIN = Path('/usr/local/envs/qe_env/bin')

RUN_ROOT   = Path('/content')
PW_CMD     = [str(QE_BIN / 'pw.x')]
PSEUDO_DIR = Path(REPO_DIR) / 'pseudo'
OUT_DIR    = RUN_ROOT / 'out'
CONV_DIR   = RUN_ROOT / 'convergence' / 'conv_thr'

for d in [OUT_DIR, CONV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('pw.x:', PW_CMD[0])

In [ ]:
from ase.build import bulk

from pw_input import (
    ControlNamelist, SystemNamelist, ElectronsNamelist,
    AtomicSpeciesCard, AtomicPositionsCard, KPointsAutoCard, PWInput,
)
from convergence_runner import (
    QERunner, RY_TO_EV,
    first_globally_converged_index,
)
from convergence_plotting import plot_conv_thr_sweep

PSEUDOS = {'Mg': 'Mg.upf', 'O': 'O.upf'}

missing = [f for f in PSEUDOS.values() if not (PSEUDO_DIR / f).is_file()]
if missing:
    raise FileNotFoundError(f'Missing pseudopotentials: {missing}')

atoms = bulk('MgO', 'rocksalt', a=4.21)
nat   = len(atoms)
print('System: MgO rocksalt, nat =', nat)

In [ ]:
def build_mgo_input(atoms, ecutwfc, nk, prefix, conv_thr):
    control   = ControlNamelist(
        calculation='scf', prefix=prefix,
        pseudo_dir=str(PSEUDO_DIR), outdir=str(OUT_DIR),
        tprnfor=True, verbosity='medium',
    )
    system    = SystemNamelist.from_atoms(atoms, ibrav=2, ecutwfc=ecutwfc)
    electrons = ElectronsNamelist(conv_thr=conv_thr)
    species   = AtomicSpeciesCard.from_atoms(atoms, PSEUDOS)
    positions = AtomicPositionsCard.from_atoms(atoms, units='crystal')
    kpoints   = KPointsAutoCard(2, nk=nk)
    return PWInput(
        control=control, system=system, electrons=electrons,
        atomic_species=species, atomic_positions=positions, k_points=kpoints,
    )

In [ ]:
# Student input — use the ecutwfc and nk you found to be converged in the
# previous notebook.
ECUTWFC   = 80
NK        = 4
FORCE_THRESHOLD_MEV_PER_ANG = 10.0

DISPLACED_ATOM_INDEX_1BASED = 1
DISPLACEMENT_ANG            = 0.01   # Å — small enough that SCF error is visible

CONV_THR_VALUES = [1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11, 1e-12, 1e-13]

# Set True to rerun pw.x even when an output file already exists.
FORCE_RERUN = True

print(f'ecutwfc = {ECUTWFC} Ry,  k-mesh = {NK}x{NK}x{NK}')
print(f'Displacement: {DISPLACEMENT_ANG:.4f} Ang along z')

In [ ]:
atoms_displaced = atoms.copy()
pos = atoms_displaced.get_positions()
pos[DISPLACED_ATOM_INDEX_1BASED - 1, 2] += DISPLACEMENT_ANG
atoms_displaced.set_positions(pos)

runner = QERunner(PW_CMD)

cases = [
    (f'conv_thr_{i}', build_mgo_input(
        atoms_displaced, ecutwfc=ECUTWFC, nk=NK,
        prefix='mgo_conv_thr', conv_thr=thr,
    ))
    for i, thr in enumerate(CONV_THR_VALUES)
]

results = runner.run_sweep(
    cases, CONV_DIR,
    force_rerun=FORCE_RERUN,
    collect_force_stress=True,
    atom_index_1based=DISPLACED_ATOM_INDEX_1BASED,
    collect_scf_correction=True,
)

In [ ]:
fz          = [r['force_z_ev_ang']      for r in results]
scf_corr    = [r['scf_correction_ev_ang'] for r in results]
time_s      = [r['wall_s']              for r in results]
dF_mev      = [abs(f - fz[-1]) * 1000  for f in fz]

idx = first_globally_converged_index(dF_mev, FORCE_THRESHOLD_MEV_PER_ANG)
conv_thr_conv = CONV_THR_VALUES[idx] if idx is not None else CONV_THR_VALUES[-1]

plot_conv_thr_sweep(
    CONV_THR_VALUES, fz, scf_corr, dF_mev, time_s,
    force_threshold_mev_ang=FORCE_THRESHOLD_MEV_PER_ANG,
)
print(f'Force converged to {FORCE_THRESHOLD_MEV_PER_ANG} meV/Å at conv_thr = {conv_thr_conv:.0e} Ry')

---

## Silicon (diamond)

Same exercise repeated for Si in the diamond structure.  Si has a norm-conserving
pseudo, so the setup is analogous to MgO.  The primitive cell (ibrav = 2, FCC)
contains two Si atoms; we displace atom 1 along z.

In [ ]:
def build_si_input(atoms, ecutwfc, nk, prefix, conv_thr):
    control   = ControlNamelist(
        calculation='scf', prefix=prefix,
        pseudo_dir=str(PSEUDO_DIR), outdir=str(OUT_DIR),
        tprnfor=True, verbosity='medium',
    )
    system    = SystemNamelist.from_atoms(atoms, ibrav=2, ecutwfc=ecutwfc)
    electrons = ElectronsNamelist(conv_thr=conv_thr)
    species   = AtomicSpeciesCard.from_atoms(atoms, PSEUDOS_SI)
    positions = AtomicPositionsCard.from_atoms(atoms, units='crystal')
    kpoints   = KPointsAutoCard(2, nk=nk)
    return PWInput(
        control=control, system=system, electrons=electrons,
        atomic_species=species, atomic_positions=positions, k_points=kpoints,
    )

In [ ]:
PSEUDOS_SI  = {'Si': 'Si.upf'}
atoms_si    = bulk('Si', 'diamond', a=5.43)

missing_si = [f for f in PSEUDOS_SI.values() if not (PSEUDO_DIR / f).is_file()]
if missing_si:
    raise FileNotFoundError(f'Missing pseudopotentials: {missing_si}')

ECUTWFC_SI   = 40
NK_SI        = 4
DISPLACEMENT_ANG_SI          = 0.01
DISPLACED_ATOM_INDEX_SI      = 1
FORCE_THRESHOLD_MEV_PER_ANG_SI = 10.0

CONV_THR_VALUES_SI = [1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11, 1e-12, 1e-13]

CONV_DIR_SI  = RUN_ROOT / 'convergence' / 'conv_thr_si'
CONV_DIR_SI.mkdir(parents=True, exist_ok=True)

FORCE_RERUN_SI = True

print(f'System: Si diamond, nat = {len(atoms_si)}')
print(f'ecutwfc = {ECUTWFC_SI} Ry,  k-mesh = {NK_SI}x{NK_SI}x{NK_SI}')
print(f'Displacement: {DISPLACEMENT_ANG_SI:.4f} Ang along z')

In [ ]:
atoms_si_displaced = atoms_si.copy()
pos_si = atoms_si_displaced.get_positions()
pos_si[DISPLACED_ATOM_INDEX_SI - 1, 2] += DISPLACEMENT_ANG_SI
atoms_si_displaced.set_positions(pos_si)

cases_si = [
    (f'conv_thr_{i}', build_si_input(
        atoms_si_displaced, ecutwfc=ECUTWFC_SI, nk=NK_SI,
        prefix='si_conv_thr', conv_thr=thr,
    ))
    for i, thr in enumerate(CONV_THR_VALUES_SI)
]

results_si = runner.run_sweep(
    cases_si, CONV_DIR_SI,
    force_rerun=FORCE_RERUN_SI,
    collect_force_stress=True,
    atom_index_1based=DISPLACED_ATOM_INDEX_SI,
    collect_scf_correction=True,
)

In [ ]:
fz_si       = [r['force_z_ev_ang']        for r in results_si]
scf_corr_si = [r['scf_correction_ev_ang'] for r in results_si]
time_si     = [r['wall_s']                for r in results_si]
dF_mev_si   = [abs(f - fz_si[-1]) * 1000 for f in fz_si]

idx_si = first_globally_converged_index(dF_mev_si, FORCE_THRESHOLD_MEV_PER_ANG_SI)
conv_thr_conv_si = CONV_THR_VALUES_SI[idx_si] if idx_si is not None else CONV_THR_VALUES_SI[-1]

plot_conv_thr_sweep(
    CONV_THR_VALUES_SI, fz_si, scf_corr_si, dF_mev_si, time_si,
    force_threshold_mev_ang=FORCE_THRESHOLD_MEV_PER_ANG_SI,
)
print(f'Si: force converged to {FORCE_THRESHOLD_MEV_PER_ANG_SI} meV/Å at conv_thr = {conv_thr_conv_si:.0e} Ry')